# Volatility Regression 01 — full-feature, next-five-minute OLS (v0.1.1)

**Question.** How much of the **volatility of the next five-minute bar** does a pure OLS on the current bar's causal features explain, and which predictors can be removed without worsening out-of-sample forecasts?

**Targets** (both volatility, one subsequent bar, never across a gap or an 08:00 session change):

- `parkinson`: $\sigma^{P}_{t+1}=\log(H_{t+1}/L_{t+1})/\sqrt{4\log 2}$ from the next perpetual **trade** bar.
- `index_abs`: $\sigma^{I}_{t+1}=|r^{\text{index}}_{t+1}|/\sqrt{2/\pi}$ from the next Deribit **index** 5m log return.

Response: $Y_{t+1}=\log(\sigma_{t+1}+\epsilon_{\text{train}})$. By default both targets use the **same rows**, so their coefficients are comparable. If a feature (e.g. trade count, spread) carries weight under `parkinson` but not under `index_abs`, its weight is measurement of the trade range, not volatility.

**Models.** Full OLS (catalog features with sufficient joint coverage) and reduced OLS (greedy backward deletion on the selection split). No HAR baseline. Two parameter-free references on the same rows: **train mean** ($R^2_{\text{OOS}}$ benchmark) and **persistence** $\hat Y_{t+1}=\log(\sigma_t+\epsilon)$.

**Splits.** Development expiries only (≤ 2026-10-07; HSMM 06 confirmation untouched), chronological by session ≈ 60/20/20. The test split is scored once.

Changes from v0.1.0 follow Volatility Regression 002, approved by Ivan on 2026-10-08.

## 1. Environment and data (DuckDB read-only)

In [ ]:
from pathlib import Path
import os, sys, json, subprocess
import numpy as np
import pandas as pd
import duckdb
from dotenv import load_dotenv

HERE = Path.cwd().resolve()
ROOT = next((p for p in [HERE, *HERE.parents] if (p / "src" / "data_engineering").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Launch from the repository root or one of its subfolders.")
sys.path.insert(0, str(ROOT / "src"))
load_dotenv(ROOT / ".env")
DATA_ROOT = os.getenv("DERIBIT_DATA_ROOT")
if not DATA_ROOT:
    raise RuntimeError("DERIBIT_DATA_ROOT must be set in .env.")
DB_PATH = Path(DATA_ROOT) / "database" / "research.duckdb"
if not DB_PATH.is_file():
    raise FileNotFoundError(DB_PATH)

from data_engineering.modeling import vol_ols_5m as mod
OUTPUT = ROOT / "notebooks" / "Volatility_Regression"

DEVELOPMENT_END = "2026-10-07"   # HSMM 06 confirmation starts after this expiry
COVERAGE_MIN = 0.50
MAX_DROPS = 15
MIN_GAIN = 0.0
REQUIRE_BOTH_TARGETS = True

with duckdb.connect(str(DB_PATH), read_only=True) as con:
    con.execute("SET TimeZone = 'UTC'")
    df = con.execute(
        """
        SELECT * FROM time_series_feature_state
        WHERE TRY_CAST(expiry_date AS DATE) <= CAST(? AS DATE)
        ORDER BY underlying, state_time
        """, [DEVELOPMENT_END]).df()
ASSETS = tuple(sorted(df["underlying"].dropna().astype(str).unique()))
print("Vol-OLS version:", mod.VOL_OLS_5M_VERSION)
print("Assets:", ASSETS, "| rows:", len(df))
print("Span:", df.state_time.min(), "to", df.state_time.max())

In [ ]:
# Synthetic alignment / estimator tests; abort on failure.
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", str(ROOT / "tests"),
                "-p", "test_vol_ols_5m.py", "-v"], cwd=ROOT, check=True)

## 2. Target-quality audit

On development rows where both targets are valid. Things to read:

- `share_zero_range`, `share_trades_le_1`: zero-range bars put $Y$ at $\log\epsilon$, a mass point far below the median.
- `share_range_le_spread`: next-bar range no larger than the spread, i.e. plausibly bounce.
- `corr_log_parkinson_log_index_abs`: agreement between the two estimators of the same volatility.
- `corr_log_parkinson_log1p_trades` vs `corr_log_index_abs_log1p_trades`: how much of the trade-range target follows trade count beyond what index volatility does.

In [ ]:
audit_rows = []
for asset in ASSETS:
    g = df[df["underlying"] == asset]
    meta, _, _ = mod.make_panel(g)
    both = (meta["sigma_next_5m"].notna() & meta["index_sigma_next_5m"].notna()).to_numpy()
    audit_rows.append({"underlying": asset, **mod.target_audit(meta, both)})
audit_df = pd.DataFrame(audit_rows)
display(audit_df.round(3))

## 3. Full and reduced OLS per asset and target

In [ ]:
stats, coverage_rows, inventory_frames, coef_rows = [], [], [], []
elim_frames, clock_frames, errors = [], [], []

for asset in ASSETS:
    g = df[df["underlying"] == asset]
    for target in mod.TARGETS:
        try:
            d = mod.prepare_asset(g, target, coverage=COVERAGE_MIN,
                                  development_end=DEVELOPMENT_END,
                                  require_both_targets=REQUIRE_BOTH_TARGETS)
            X, y, names = d["X"], d["y"], d["features"]
            tr, sel, tst = (d[s] & d["common"] for s in ("train", "selection", "test"))
            if tr.sum() <= len(names) + 20 or sel.sum() < 20 or tst.sum() < 20:
                raise ValueError(f"Insufficient common rows: train={tr.sum()}, "
                                 f"selection={sel.sum()}, test={tst.sum()}, p={len(names)}")
            mu = d["train_mean"]
            full = mod.fit_ols(X, y, tr, names)
            reduced_names, hist = mod.backward_validation(
                X, y, tr, sel, names, max_drops=MAX_DROPS, min_improvement=MIN_GAIN)
            reduced = mod.fit_ols(X, y, tr, reduced_names)

            for split, mask in (("train", tr), ("selection", sel), ("test", tst)):
                idx = np.flatnonzero(mask)
                preds = {
                    "train_mean": np.full(len(idx), mu),
                    "persistence": d["persistence"][idx],
                    "full_ols": full.predict(X.iloc[idx]),
                    "reduced_ols": reduced.predict(X.iloc[idx]),
                }
                for model, pred in preds.items():
                    p = {"full_ols": len(full.names), "reduced_ols": len(reduced.names)}.get(model, 0)
                    cond = {"full_ols": full.condition_number,
                            "reduced_ols": reduced.condition_number}.get(model, np.nan)
                    stats.append({"underlying": asset, "target": target, "model": model,
                                  "features": p, "condition_number": cond,
                                  "epsilon_train": d["epsilon"],
                                  **mod.score(y[idx], pred, mu, split)})

            for fit_name, fit in (("full_ols", full), ("reduced_ols", reduced)):
                for name, beta in zip(fit.names, fit.beta[1:]):
                    coef_rows.append({"underlying": asset, "target": target, "model": fit_name,
                                      "feature": name, "beta_per_train_sd": float(beta),
                                      "condition_number": fit.condition_number})

            inv = d["inventory"].copy()
            inv["underlying"], inv["target"] = asset, target
            inv["train_coverage"] = inv["name"].map(d["coverage"])
            inv["full_ols"] = inv["name"].isin(names)
            inv["reduced_ols"] = inv["name"].isin(reduced_names)
            inv["low_coverage"] = inv["name"].isin(d["excluded_coverage"])
            inv["low_joint_coverage"] = inv["name"].isin(d["excluded_joint_coverage"])
            inv["aliased_or_constant"] = inv["name"].isin(d["excluded_aliases"])
            inventory_frames.append(inv)

            coverage_rows.append({
                "underlying": asset, "target": target,
                "target_valid_train": int(d["train"].sum()),
                "scored_train": int(tr.sum()), "scored_selection": int(sel.sum()),
                "scored_test": int(tst.sum()), "n_sessions": len(d["sessions"]),
                "catalog_columns": d["X"].shape[1], "full_p": len(names),
                "reduced_p": len(reduced_names), "max_drops_reached": len(hist) >= MAX_DROPS,
                "removed_for_coverage": len(d["excluded_coverage"]),
                "removed_for_joint_coverage": len(d["excluded_joint_coverage"]),
                "removed_for_alias": len(d["excluded_aliases"])})
            cc = mod.clock_coverage(d)
            cc["underlying"], cc["target"] = asset, target
            clock_frames.append(cc)
            if len(hist):
                hist["underlying"], hist["target"] = asset, target
                elim_frames.append(hist)
            print(f"{asset:>6} {target:<10} n={int(tr.sum())}/{int(sel.sum())}/{int(tst.sum())} "
                  f"p={len(names)} -> {len(reduced_names)}")
        except Exception as exc:
            errors.append({"underlying": asset, "target": target,
                           "reason": f"{type(exc).__name__}: {exc}"})
            print(f"{asset:>6} {target:<10} NOT FIT: {errors[-1]['reason']}")

metrics_df = pd.DataFrame(stats)
coverage_df = pd.DataFrame(coverage_rows)
inventory_df = pd.concat(inventory_frames, ignore_index=True) if inventory_frames else pd.DataFrame()
coefficients_df = pd.DataFrame(coef_rows)
elimination_df = pd.concat(elim_frames, ignore_index=True) if elim_frames else pd.DataFrame()
clock_df = pd.concat(clock_frames, ignore_index=True) if clock_frames else pd.DataFrame()
errors_df = pd.DataFrame(errors)
display(coverage_df)
if not errors_df.empty:
    display(errors_df)

## 4. Test scores (scored once)

RMSE ratio is relative to persistence on the same rows: below 1 means the model beats "next bar = current bar".

In [ ]:
if not metrics_df.empty:
    test = metrics_df[metrics_df["split"] == "test"].copy()
    ref = test[test["model"] == "persistence"].set_index(["underlying", "target"])["log_rmse"]
    test["rmse_vs_persistence"] = test["log_rmse"] / test.set_index(
        ["underlying", "target"]).index.map(ref).to_numpy()
    display(test.pivot_table(index=["underlying", "target"], columns="model",
                             values=["r2_vs_train_mean", "rmse_vs_persistence"]).round(3))

## 5. Coefficients, elimination history, clock coverage

Coefficients are per one training standard deviation of each feature. With many correlated regressors they are **descriptive only**; check the condition number. Compare each feature across the two targets.

In [ ]:
if not coefficients_df.empty:
    red = coefficients_df[coefficients_df["model"] == "reduced_ols"]
    display(red.pivot_table(index=["underlying", "feature"], columns="target",
                            values="beta_per_train_sd").round(3))
if not elimination_df.empty:
    display(elimination_df)
if not clock_df.empty:
    print("Scored share of target-valid rows by UTC hour (train split):")
    display(clock_df[clock_df["split"] == "train"].pivot_table(
        index="utc_hour", columns=["underlying", "target"], values="scored_share").round(2))

## 6. Save outputs (no database writes)

In [ ]:
outputs = {
    "01_target_audit.csv": audit_df,
    "01_scores.csv": metrics_df,
    "01_coverage.csv": coverage_df,
    "01_clock_coverage.csv": clock_df,
    "01_feature_inventory.csv": inventory_df,
    "01_coefficients.csv": coefficients_df,
    "01_elimination_history.csv": elimination_df,
    "01_fit_errors.csv": errors_df,
}
for name, table in outputs.items():
    table.to_csv(OUTPUT / name, index=False)
    print(name, "rows:", len(table))
summary = {
    "version": mod.VOL_OLS_5M_VERSION,
    "targets": {"parkinson": "next 5m perpetual trade-range Parkinson volatility",
                "index_abs": "|next 5m index log return| / sqrt(2/pi)"},
    "transformed_target": "log(sigma_next_5m + train_only_epsilon)",
    "require_both_targets": REQUIRE_BOTH_TARGETS,
    "references": ["train_mean", "persistence"],
    "r2_benchmark": "training mean",
    "development_last_expiry": DEVELOPMENT_END,
    "coverage_threshold": COVERAGE_MIN,
    "max_backward_drops": MAX_DROPS,
    "fitted": [f"{r['underlying']}/{r['target']}" for r in coverage_rows],
    "failed": errors,
    "claim": "development experiment only; no held-out confirmation",
}
(OUTPUT / "01_run_metadata.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
# Remove v0.1.0 output superseded by 01_coefficients.csv, if present.
old = OUTPUT / "01_full_ols_coefficients.csv"
if old.exists():
    old.unlink()
print("Done. Commit the notebook and 01_* outputs.")

## Interpretation contract

1. Exclusion for coverage or collinearity is **not** evidence of zero predictive content.
2. Backward deletion uses the selection split only (≈3 sessions); removals with tiny gains are likely noise. The test split is a one-time diagnostic, not a tuning input.
3. A feature whose coefficient appears under `parkinson` but not under `index_abs` is measuring the trade range (trade count, bounce), not volatility.
4. Rows are complete-case; check `01_clock_coverage.csv` for hours (08:00–09:00 UTC, settlement window) that the scored sample under-represents.
5. Research diagnostics only; HSMM 06 frozen spec and confirmation window untouched.